# C1_07_Class_Weighting
- X = X_flat_c1_v2.npy
- Y = y_labels_c1_v2.csv
- Built by Garmin_C1_DataBuild

### Does weighting the classes help? Dense 32-16 on `prev1`, with and without class weights

Recovery is only 2% of windows, so this tests whether giving the rarer classes more weight in the loss helps. The weights are "balanced": each class is weighted in inverse proportion to how often it appears in the TRAIN windows.

The model is the chosen dense network from C1_05_Dense_vs_GRU (32-16, previous window as input), trained with the same settings (batch 256, up to 40 epochs, early stopping with patience 5) on seeds 42, 7 and 123. The unweighted runs are read from the saved results of C1_05_Dense_vs_GRU (`c1_context_final_2026-10-04`), so both are scored on the same VAL windows. The test set is not used.

## 1. Setup and configuration

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight

from tensorflow import keras

DATA_DIR = Path(r"C:\dev\UoW\DATAX504\Final Assignment\garmin_data\Processed")
X_FLAT_PATH = DATA_DIR / "X_flat_c1_v2.npy"
Y_LABELS_PATH = DATA_DIR / "y_labels_c1_v2.csv"
UNWEIGHTED_DIR = DATA_DIR / "c1_context_final_2026-10-04"     # saved results of the unweighted dense networks (C1_05_Dense_vs_GRU)
RESULTS_DIR = DATA_DIR / "c1_class_weighting"                 # where this notebook saves its predictions
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SECONDS = 30
SEQ_COLS = 120               # first 120 columns of X_flat: HR, speed, cadence, temperature (30 each); column 120 = distance
COMMON_LAGS = [1, 2]         # windows must have both earlier windows (same set as C1_05_Dense_vs_GRU)
CONTEXT_LAG = 1              # the earlier window added as input (1 = the window just before)

DENSE_NAME = "32-16"
HIDDEN = [32, 16]            # hidden layer sizes; dropout follows every hidden layer except the last
DROPOUT = 0.2
SEEDS = [42, 7, 123]
BATCH = 256
EPOCHS = 40
PATIENCE = 5

## 2. Load data and build the common window set

Each window's earlier windows are found by matching `(file_path, start_s - 30 × lag)`. The first windows of each run have no earlier window, so they are left out. Every model is scored on the same windows, those that have all the earlier windows. The code checks that each earlier window comes from the same file and exactly the expected time before.

In [2]:
X_flat = np.load(X_FLAT_PATH).astype("float32")
y_df = pd.read_csv(Y_LABELS_PATH, parse_dates=["date"])
assert len(X_flat) == len(y_df), "X and Y row counts differ"
y_df["start_s"] = y_df["start_s"].astype(int)
print(f"X_flat {X_flat.shape}, {len(y_df)} labelled windows")

row_lookup = pd.Series(np.arange(len(y_df)), index=pd.MultiIndex.from_arrays([y_df["file_path"], y_df["start_s"]]))
assert row_lookup.index.is_unique, "duplicate (file_path, start_s) keys"

lag_rows = {}
for lag in COMMON_LAGS:
    keys = pd.MultiIndex.from_arrays([y_df["file_path"], y_df["start_s"] - WINDOW_SECONDS * lag])
    lag_rows[lag] = row_lookup.reindex(keys).to_numpy()          # NaN where that earlier window does not exist

common = np.ones(len(y_df), dtype=bool)
for lag in COMMON_LAGS:
    common &= ~np.isnan(lag_rows[lag])
print(f"Windows with every earlier window available: {int(common.sum())} of {len(y_df)} ({100 * common.mean():.1f}%)")

for lag in COMMON_LAGS:
    r = lag_rows[lag][common].astype(int)
    assert (y_df["file_path"].to_numpy()[r] == y_df["file_path"].to_numpy()[common]).all(), "earlier window from a different file"
    assert (y_df["start_s"].to_numpy()[r] == y_df["start_s"].to_numpy()[common] - WINDOW_SECONDS * lag).all(), "earlier window at the wrong time"
print("OK: every earlier window is from the same file and exactly the expected time before.")

classes = np.array(sorted(y_df["c1_target"].unique()))
class_to_int = {c: i for i, c in enumerate(classes)}
n_classes = len(classes)
y_all = np.array([class_to_int[c] for c in y_df["c1_target"]])
print("Classes:", {str(c): i for c, i in class_to_int.items()})

X_flat (197341, 121), 197341 labelled windows
Windows with every earlier window available: 193591 of 197341 (98.1%)
OK: every earlier window is from the same file and exactly the expected time before.
Classes: {'Base': 0, 'Declining': 1, 'Improving': 2, 'Recovery': 3}


## 3. Split by run

The split is the same as in the baselines. The test runs are set aside and never built. The code checks that the validation windows are the same as the saved ones from C1_05_Dense_vs_GRU, so the weighted and unweighted runs can be compared directly.

In [3]:
run_idx_all = y_df["run_idx"].to_numpy()
unique_runs = np.unique(run_idx_all)
train_runs, held_out_runs = train_test_split(unique_runs, test_size=0.2, random_state=42)
trainval_idx = np.where(np.isin(run_idx_all, train_runs))[0]
trainval_run_idx = run_idx_all[trainval_idx]
fit_runs, val_runs = train_test_split(np.unique(trainval_run_idx), test_size=0.15, random_state=42)

fit_idx = trainval_idx[np.isin(trainval_run_idx, fit_runs) & common[trainval_idx]]
val_idx = trainval_idx[np.isin(trainval_run_idx, val_runs) & common[trainval_idx]]
y_fit, y_val = y_all[fit_idx], y_all[val_idx]
print(f"Fit: {len(fit_idx)} windows | Val: {len(val_idx)} windows from {len(val_runs)} runs (test runs excluded, never built)")

saved_val_idx = np.load(UNWEIGHTED_DIR / "val_idx.npy")
assert np.array_equal(val_idx, saved_val_idx), "validation windows differ from the saved unweighted runs"
print("OK: same validation windows as the unweighted runs.")

Fit: 130416 windows | Val: 24095 windows from 225 runs (test runs excluded, never built)
OK: same validation windows as the unweighted runs.


## 4. Input features

The same input as the dense networks in C1_05_Dense_vs_GRU: the current window's 121 values plus the earlier window's 120 readings, 241 features, scaled using the TRAIN windows only.

In [4]:
def build_dense_features(idx):
    return np.concatenate([X_flat[idx], X_flat[lag_rows[CONTEXT_LAG][idx].astype(int), :SEQ_COLS]], axis=1)

Xf_raw, Xv_raw = build_dense_features(fit_idx), build_dense_features(val_idx)
scaler = StandardScaler().fit(Xf_raw)
X_fit = scaler.transform(Xf_raw).astype("float32")
X_val = scaler.transform(Xv_raw).astype("float32")
print(f"Dense input: {X_fit.shape[1]} features")

Dense input: 241 features


## 5. Dense 32-16 with balanced class weights (3 seeds)

The loss counts each window in proportion to its class weight, so mistakes on the rare classes cost more. Everything else is the same as the unweighted network.

In [5]:
weights = compute_class_weight("balanced", classes=np.arange(n_classes), y=y_fit)
class_weight = dict(enumerate(weights))
print("Class weights:", {str(c): round(float(w), 2) for c, w in zip(classes, weights)})


def build_dense(n_features):
    layers = [keras.layers.Input(shape=(n_features,))]
    for i, units in enumerate(HIDDEN):
        layers.append(keras.layers.Dense(units, activation="relu"))
        if i < len(HIDDEN) - 1:
            layers.append(keras.layers.Dropout(DROPOUT))
    layers.append(keras.layers.Dense(n_classes, activation="softmax"))
    model = keras.Sequential(layers)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


weighted_preds = []
for seed in SEEDS:
    keras.utils.set_random_seed(seed)
    model = build_dense(X_fit.shape[1])
    t0 = time.time()
    history = model.fit(
        X_fit, y_fit, validation_data=(X_val, y_val),
        epochs=EPOCHS, batch_size=BATCH, verbose=0, class_weight=class_weight,
        callbacks=[keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True)],
    )
    pred = model.predict(X_val, batch_size=1024, verbose=0).argmax(axis=1)
    np.save(RESULTS_DIR / f"pred_weighted_seed{seed}.npy", pred)
    weighted_preds.append(pred)
    best_epoch = int(np.argmin(history.history["val_loss"])) + 1
    print(f"seed {seed}: val accuracy {accuracy_score(y_val, pred):.3f} | macro-F1 {f1_score(y_val, pred, average='macro'):.3f} "
          f"| best epoch {best_epoch} | {(time.time() - t0) / 60:.1f} min", flush=True)
    keras.backend.clear_session()

Class weights: {'Base': 0.64, 'Declining': 0.78, 'Improving': 0.92, 'Recovery': 13.03}
seed 42: val accuracy 0.584 | macro-F1 0.605 | best epoch 40 | 2.0 min

seed 7: val accuracy 0.584 | macro-F1 0.601 | best epoch 28 | 1.6 min
seed 123: val accuracy 0.575 | macro-F1 0.593 | best epoch 21 | 1.3 min


## 6. Comparison: weighted vs unweighted (VAL windows)

Each row is the mean over the 3 seeds. The unweighted predictions are the saved ones from C1_05_Dense_vs_GRU, and both are scored with the same code on the same windows.

In [6]:
def mean_scores(preds):
    rows = []
    for p in preds:
        row = {"accuracy": accuracy_score(y_val, p), "macro_f1": f1_score(y_val, p, average="macro")}
        precision = precision_score(y_val, p, average=None, labels=np.arange(n_classes), zero_division=0)
        recall = recall_score(y_val, p, average=None, labels=np.arange(n_classes), zero_division=0)
        for c, pv, rv in zip(classes, precision, recall):
            row[f"precision_{c}"] = pv
            row[f"recall_{c}"] = rv
        rows.append(row)
    return pd.DataFrame(rows).mean()


unweighted_preds = [np.load(UNWEIGHTED_DIR / f"pred_dense_{DENSE_NAME}_seed{s}.npy") for s in SEEDS]

# Check the saved predictions reproduce the accuracy logged by C1_05_Dense_vs_GRU
logged = pd.read_csv(UNWEIGHTED_DIR / "results.csv")
logged_acc = logged[(logged["stage"] == "dense") & (logged["name"] == DENSE_NAME)]["val_acc"].mean()
unweighted_acc = mean_scores(unweighted_preds)["accuracy"]
assert abs(unweighted_acc - logged_acc) < 1e-6, "saved predictions do not match the logged accuracy"
print(f"Check: unweighted accuracy from the saved predictions ({unweighted_acc:.3f}) matches results.csv ({logged_acc:.3f}).\n")

comparison = pd.DataFrame({"Unweighted": mean_scores(unweighted_preds), "Class-weighted": mean_scores(weighted_preds)}).T
comparison.to_csv(RESULTS_DIR / "comparison.csv")
print(f"Dense {DENSE_NAME} with the previous window, mean of {len(SEEDS)} seeds (validation):")
print(comparison.to_string(float_format=lambda v: f"{v:.3f}"))

Check: unweighted accuracy from the saved predictions (0.599) matches results.csv (0.599).

Dense 32-16 with the previous window, mean of 3 seeds (validation):
                accuracy  macro_f1  precision_Base  recall_Base  precision_Declining  recall_Declining  precision_Improving  recall_Improving  precision_Recovery  recall_Recovery
Unweighted         0.599     0.634           0.571        0.674                0.627             0.544                0.609             0.544               0.795            0.731
Class-weighted     0.581     0.600           0.586        0.567                0.599             0.567                0.566             0.590               0.503            0.974
